# M65c — calibrated A2 preservation control

Revision: `M65C-A2-SCALED-PRESERVATION-2026-10-06-r1` (2026-10-06).

Run the six sections below in order in a **Colab GPU runtime** (A100 recommended). This is one controlled epoch, not external-teacher distillation. Original A2 remains selected. No phone is needed.

New hypothesis: scale-normalized preservation, including depth log uncertainty, may retain A2 more effectively. One scalar is calibrated on 64 training images with zero updates at the reviewed M65 endpoint. Training still starts from original A2 epoch130. No validation-based scalar search, architecture change, teacher download, or extra epochs.

Required Drive files: the completed M65 manifest/checkpoint and M65b `gradient_r1` original/control reports and runtime receipt. Keep the original A2 checkpoint at the path recorded in M65. This notebook restores source/runtime/data without rerunning M64 or M65b.

After a runtime restart, rerun sections 1–4. Completed evaluations and the completed one-epoch checkpoint are reused after validation. A partially interrupted epoch restarts from original A2, not from its last batch. Changed GPU/environment/source requires a new RUN_ID and fresh preflight; do not rewrite old evidence.

## 1. Settings, Drive, helpers, and latest main

Check paths below. The printed revision confirms which notebook is running.


In [ ]:
# M65C-A2-SCALED-PRESERVATION-2026-10-06-r1 — self-contained six-section notebook
from pathlib import Path
from collections import deque
import ast, json, os, runpy, shlex, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

REVISION = 'M65C-A2-SCALED-PRESERVATION-2026-10-06-r1'
RUN_ID = 'm65c_a2_preservation_r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
A2_REPO = Path('/content/MonoDETR_M65C_A2')
M65_OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m65_a2_preservation/m65_a2_preservation_r1')
M65_MANIFEST = M65_OUTPUT/'m65_manifest.json'
M65B_OUTPUT = M65_OUTPUT/'diagnostics_m65b/gradient_r1'
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m65c_scaled_preservation')/RUN_ID
DATASET = Path('/content/kitti_m65c')
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
PYTHON = Path('/content/m65c_cuda130_venv/bin/python')
CUDA = Path('/usr/local/cuda-13.0')
if not (CUDA/'bin/nvcc').is_file(): CUDA = Path('/usr/local/cuda')
MANIFEST = OUTPUT/'m65c_manifest.json'
SETUP_READY = MANIFEST_READY = TRAIN_COMPLETE = False
TRAIN_READY = False
ENV = os.environ.copy()
OUTPUT.mkdir(parents=True,exist_ok=True)
print(REVISION, '\nOutput:', OUTPUT, flush=True)

def run_logged(command, name, cwd=None):
    command = [str(x) for x in command]
    log_path = OUTPUT/'colab_logs'/f'{name}.log'
    log_path.parent.mkdir(parents=True,exist_ok=True)
    tail = deque(maxlen=60)
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True)
    with log_path.open('a') as log:
        log.write('\nCOMMAND: '+shlex.join(command)+'\n'); log.flush()
        with subprocess.Popen(command,cwd=str(cwd) if cwd else None,env=ENV,
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
            status = process.wait()
        log.write(f'\nEXIT CODE: {status}\n'); log.flush()
    if status: raise RuntimeError(f'Exit {status}; full log={log_path}\n'+'\n'.join(tail))

def checkout(path,url,commit):
    if '[' in url or ']' in url or not url.startswith('https://github.com/'):
        raise ValueError('Use a plain public GitHub URL, not Markdown')
    if not path.exists():
        run_logged(['git','clone','--no-checkout',url,path],f'clone_{path.name}')
        run_logged(['git','checkout','--detach',commit],f'pin_{path.name}',path)
    if not (path/'.git').is_dir(): raise RuntimeError(f'Preserve {path}; choose a fresh checkout path')
    head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=path,text=True).strip()
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=path,text=True).strip()
    if head != commit or origin.rstrip('/') != url:
        raise RuntimeError(f'Unexpected checkout identity: {path}; do not reset it')

def workflow(stage,*extra):
    command = [PYTHON,'-X','faulthandler','-u',SCRIPT,stage]
    command += ['--output',OUTPUT] if stage in ('prepare','bundle') else ['--manifest',MANIFEST]
    name = 'm65c_'+stage
    if '--model' in extra: name += '_'+str(extra[extra.index('--model')+1])
    run_logged(command+list(extra),name,MOBILE_REPO)

if not MOBILE_REPO.exists():
    run_logged(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone_mobile')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branch work')
run_logged(['git','pull','--ff-only'],'update_mobile',MOBILE_REPO)
SCRIPT = MOBILE_REPO/'scripts/m65c_a2_preservation_control.py'
if not SCRIPT.is_file(): raise FileNotFoundError('M65c is not published yet; reload the latest main notebook.')
# Inspect the revision without importing model packages into the Colab kernel.
revisions = [ast.literal_eval(node.value) for node in ast.parse(SCRIPT.read_text()).body
             if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='REVISION' for t in node.targets)]
if revisions != [REVISION]: raise RuntimeError('Notebook/script revisions differ; reload the latest notebook')


## 2. Isolated CUDA13 runtime, fresh A2 source, and tests

Uses the working M64/M65 runtime helpers; their names do not change the experiment. Only A2 is built. The kernel packages are not replaced. Toolkit/GPU failure stops here; do not bypass it.


In [ ]:
SETUP_READY = MANIFEST_READY = TRAIN_READY = TRAIN_COMPLETE = False
if not SCRIPT.is_file(): raise RuntimeError('Run section 1 first')
helpers = runpy.run_path(str(MOBILE_REPO/'scripts/setup_m64_runtime.py'),run_name='shared_cuda13_helpers')
ENV = helpers['runtime_env'](CUDA,PYTHON.parent)
run_logged([sys.executable,'-u',MOBILE_REPO/'scripts/setup_m64_runtime.py',
            '--venv',PYTHON.parent.parent,'--cuda',CUDA,'--receipt',OUTPUT/'m65c_runtime_receipt.json'],
           'm65c_runtime',MOBILE_REPO)
checkout(A2_REPO,'https://github.com/ZrrSkywalker/MonoDETR.git','6994b9f512400b258c6edb75f77423beb9c126f2')
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_mobilenetv4.py'):
    run_logged([PYTHON,MOBILE_REPO/'scripts'/patch,'--monodetr-repo',A2_REPO],patch.removesuffix('.py'),MOBILE_REPO)
run_logged([PYTHON,MOBILE_REPO/'scripts/patch_m64_inference_sources.py','--repo',A2_REPO,'--role','a2'],'m65c_patch_a2',MOBILE_REPO)
run_logged([PYTHON,'-X','faulthandler','-u',MOBILE_REPO/'scripts/build_m64_attention.py',
            '--repo',A2_REPO,'--role','a2'],'m65c_build_a2',MOBILE_REPO)
for test in ('test_m64_teacher_qualification.py','test_m65_preservation_control.py','test_m65c_preservation_control.py'):
    run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p',test,'-v'],test.removesuffix('.py'),MOBILE_REPO)
SETUP_READY = True
print('Section 2 passed: isolated runtime, original A2 source, local attention build, CPU tests.')


## 3. Restore exact KITTI data and freeze M65c provenance

Existing Drive evidence is read-only. M65c requires the exact reviewed M65/M65b result; it does not require historical temporary Colab checkouts. Missing files stop with their full path.


In [ ]:
MANIFEST_READY = TRAIN_READY = TRAIN_COMPLETE = False
if not SETUP_READY: raise RuntimeError('Run sections 1 and 2 first')
for required in (M65_MANIFEST,M65B_OUTPUT/'original_gradients.json',
                 M65B_OUTPUT/'control_gradients.json',M65B_OUTPUT/'runtime_receipt.json'):
    if not required.is_file(): raise FileNotFoundError(required)
candidates = ['/content/kitti_m65','/content/kitti_m65b','/content/kitti_m64',
              '/content/kitti_m62','/content/kitti','/content/monodetr_kitti',
              '/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti']
command = [PYTHON,MOBILE_REPO/'scripts/restore_m63_data.py','--dataset',DATASET,'--splits',SPLITS]
for candidate in candidates: command += ['--candidate',candidate]
run_logged(command,'m65c_restore_data',MOBILE_REPO)
workflow('prepare','--m65-manifest',M65_MANIFEST,'--m65b-output',M65B_OUTPUT,
         '--a2-repo',A2_REPO,'--dataset-root',DATASET)
manifest = json.loads(MANIFEST.read_text())
assert manifest['recipe']['epochs']==1 and manifest['kd_authorized'] is False
assert len(set(manifest['calibration_ids']))==64
MANIFEST_READY = True
print('Section 3 passed: original A2 and exact Chen split identities frozen.')


## 4. Reproduce A2, calibrate once on training data, and CUDA smoke

Full 3,769-image baseline first. Then measure gradients on 64 training images (32 Pedestrian-containing + 32 Vehicle-only). Calibration performs **zero updates** at the reviewed M65 endpoint and sets one coefficient: `0.25 / median(preservation gradient norm / GT gradient norm)`. This is a design hypothesis, not an optimality claim. Invalid coverage/scalar stops; no clamping or automatic search.

The smoke starts from original A2 and proves finite uncertainty/GT gradients without updates. No validation score is used to select the coefficient.


In [ ]:
TRAIN_READY = TRAIN_COMPLETE = False
if not MANIFEST_READY: raise RuntimeError('Run sections 1–3 first')
workflow('infer','--model','baseline')
workflow('metrics','--model','baseline')
workflow('baseline')
workflow('calibrate')
workflow('smoke')
calibration = json.loads((OUTPUT/'m65c_calibration.json').read_text())
smoke = json.loads((OUTPUT/'m65c_training_smoke.json').read_text())
assert calibration['optimizer_steps']==0 and smoke['optimizer_steps']==0
assert smoke['passed'] and not smoke['external_kd_enabled']
assert smoke['calibration_sha256']==calibration['report_sha256']
TRAIN_READY = True
print('Section 4 passed. Frozen preservation coefficient:',calibration['coefficient'])
print('Only the specified one-epoch control may run; training starts again from original A2.')


## 5. One epoch from original A2

All native GT terms for both classes remain active. BatchNorm affine/buffers and dropout remain frozen; LR `1e-6`, batch 4, 928 updates, no AMP. No external teacher is used.

A completed epoch checkpoint is atomic and reusable. If interrupted mid-epoch, rerunning restarts this bounded epoch deterministically from original A2. It is not batch-level resume. Do not add epochs.


In [ ]:
if not TRAIN_READY: raise RuntimeError('Run sections 1–4; do not bypass failed preflight')
TRAIN_COMPLETE = False
workflow('train')
summary = json.loads((OUTPUT/'m65c_training_summary.json').read_text())
assert summary['complete'] and summary['completed_epochs']==1 and summary['optimizer_steps']==928
assert summary['running_buffers_unchanged'] and summary['anchor_unchanged']
assert not summary['external_kd_enabled']
TRAIN_COMPLETE = True
print('One epoch complete. Run section 6 for full accuracy review; do not enable KD.')


## 6. Full evaluation, stop for review, and return bundle

Preservation gate: no more than **0.15 AP points** lost on any of the four moderate 3D/BEV metrics, and no more than **0.005 nearby recall** lost for either class. Historical product gates are reported separately and unchanged. A stable control is not proof of improvement or permission for teacher KD.

Return `m65c_results.zip`. Large model weights and prediction files are excluded. You can also run this section after an earlier failure to bundle partial diagnostics. It always attempts bundling even if evaluation fails.


In [ ]:
try:
    if TRAIN_COMPLETE:
        workflow('infer','--model','control')
        workflow('metrics','--model','control')
        workflow('review')
    else:
        print('Partial diagnostics only; no completed-control claim.')
finally:
    workflow('bundle')
    print('Return:',OUTPUT/'m65c_results.zip')
print('STOP FOR REVIEW. Original A2 remains selected. No extra epochs, KD, compression, or phone deployment.')
